# Lab 21: How to call any model through one interface with LiteLLM

**Notebook 1 of 2: meet LiteLLM by naming a kitten**

## Goal

Send a question to a model through LiteLLM instead of through the vendor's own client, and see what the layer in
between gives us: one call for every vendor, a cost for every reply, names of our own for the models, and a second
model that answers when the first one cannot be reached. The job is small on purpose. We have a new kitten, and she
needs a name. This notebook is the playground; notebook 2 takes LiteLLM to
the Chinook support desk.

## What you will learn

- How `litellm.completion` mirrors the call of OpenAI's own client, and what the `openai/` in front of a model
  name says.
- How to read what a reply cost with `completion_cost`, and why that number is an estimate from a price table.
- How to ask several models the same question in one loop, and what `drop_params` is for.
- How to receive a reply piece by piece, and how to let the model ask for a tool that our own code runs.
- How to give models names of our own with a `Router`, and read from a reply which deployment wrote it.
- How a fallback keeps an answer coming when the first deployment cannot be reached.
- How to put the same names behind one address with the LiteLLM gateway, and call it with the plain OpenAI client.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name and the prices from
`models.env`, and takes the key from `.env` without printing it (lab 00 explains it line by line).

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


---

## Part A: Send one question through OpenAI's client and through LiteLLM

We have a new kitten. She is grey, she climbs the curtains, and she has no name yet. So this is our question for
the model:

> Suggest one name for a grey kitten who climbs the curtains. Reply with the name only.

We send it twice: first with OpenAI's own client, then with LiteLLM. With the two calls side by side we can see
what LiteLLM is: a library that takes one kind of call and passes it on to the vendor we name.

### The call with OpenAI's own client

`client.chat.completions.create` is OpenAI's Chat Completions API. Many model vendors and tools have copied the
shape of this call, and LiteLLM is built around it.

- `model` is the model's name, from `models.env`.
- `messages` is the conversation so far, as a list. Ours has one message, from the user.
- `reasoning_effort` passes the setting the setup cell printed. With `none` the model answers without reasoning
  first. `EFFORT or omit` leaves the field out when the setting is empty, for models that do not take it.
- `timeout=60` gives up on a request after 60 seconds, so a request that gets stuck cannot freeze a cell.

The reply's text is in `response.choices[0].message.content`.

In [2]:
from openai import OpenAI, omit

QUESTION = "Suggest one name for a grey kitten who climbs the curtains. Reply with the name only."
kitten = [{"role": "user", "content": QUESTION}]

client = OpenAI(timeout=60)

openai_response = client.chat.completions.create(
    model=MODEL,
    messages=kitten,
    reasoning_effort=EFFORT or omit,
)

print(openai_response.choices[0].message.content)

Smokey Drapes


A name for the kitten: *Smokey Drapes* in our run. The model writes a new reply on every call, so yours will
probably be another name.

### The same call with LiteLLM

`litellm.completion` takes the same arguments. One thing changes: the model's name gets a prefix. In
`"openai/" + MODEL`, the part before the slash names the vendor LiteLLM should send the call to, and the part after
it is the model's name at that vendor. LiteLLM reads the key from the same place the OpenAI client does, the
environment variable `OPENAI_API_KEY`.

Two settings come first. `litellm.request_timeout = 60` is the same 60-second limit as above.
`litellm.suppress_debug_info = True` switches off a notice that LiteLLM prints with every error, so that the errors
we cause on purpose later are easy to read.

LiteLLM leaves `reasoning_effort` out by itself when the setting is empty, so we pass `EFFORT` as it is.

In [3]:
import litellm

litellm.request_timeout = 60
litellm.suppress_debug_info = True

litellm_response = litellm.completion(
    model="openai/" + MODEL,
    messages=kitten,
    reasoning_effort=EFFORT,
)

print(litellm_response.choices[0].message.content)

Smokeyapes


Another name, *Smokeyapes* in our run. We read the text with the same line as before,
`choices[0].message.content`.

### Compare the two replies field by field

Reading the text the same way could be a coincidence. We print the same six things from both replies: the type of
the Python object, the text, the model that answered, why the reply ended, and the two token counts.

In [4]:
for label, response in [("OpenAI client", openai_response), ("LiteLLM", litellm_response)]:
    print(label)
    print("  type:         ", type(response).__name__)
    print("  text:         ", response.choices[0].message.content)
    print("  model:        ", response.model)
    print("  finish_reason:", response.choices[0].finish_reason)
    print("  input tokens: ", response.usage.prompt_tokens)
    print("  output tokens:", response.usage.completion_tokens)

OpenAI client
  type:          ChatCompletion
  text:          Smokey Drapes
  model:         gpt-6-luna
  finish_reason: stop
  input tokens:  24
  output tokens: 8
LiteLLM
  type:          ModelResponse
  text:          Smokeyapes
  model:         gpt-6-luna
  finish_reason: stop
  input tokens:  24
  output tokens: 7


Read the two blocks line against line:

- **`type`** is the only line that differs in kind. OpenAI's client returns a `ChatCompletion`. LiteLLM returns its
  own object, a `ModelResponse`.
- **Every other line was read with the same code,** and both replies had the field: `choices[0].message.content`,
  `model`, `choices[0].finish_reason` (`stop` means the model finished its answer), `usage.prompt_tokens` and
  `usage.completion_tokens`. Both calls sent the same 24 input tokens, because the question was the same.

**Key lesson:** a `ModelResponse` is built to have the fields of OpenAI's reply. A vendor's own client returns the
vendor's own shape: Google's and Anthropic's replies have other field names. Through LiteLLM a reply from any of
them arrives in this one shape, so the code that reads it does not change when the vendor does.

![Our code calls litellm.completion with a model name and messages. LiteLLM passes the call on to OpenAI, Gemini or Anthropic, chosen by the prefix of the model name, and returns the reply in one shape.](resources/lab21_one_interface.png)

Light green is our own code, the dark boxes are models, white is LiteLLM.

**Checkpoint A.** Both calls print a name for the kitten. The first reply is a `ChatCompletion` and the second a
`ModelResponse`, and the other five lines print from both with the same code.

---

## Part B: Read what a reply cost, and where the price comes from

A vendor bills by the token. Every reply carries its token counts in `usage`, so we can price it ourselves. We do
that once by hand, and then let LiteLLM do it.

### The cost by hand, from `models.env`

`models.env` holds two prices for our model, in US dollars per million tokens: one for input tokens (what we sent)
and one for output tokens (what the model wrote).

$$\text{cost} = \frac{\text{input tokens} \times \text{input price} + \text{output tokens} \times \text{output price}}{1{,}000{,}000}$$

In [5]:
PRICE_IN = float(settings["PRICE_INPUT_PER_MILLION"])
PRICE_OUT = float(settings["PRICE_OUTPUT_PER_MILLION"])

usage = litellm_response.usage
by_hand = (usage.prompt_tokens * PRICE_IN + usage.completion_tokens * PRICE_OUT) / 1_000_000

print("input tokens: ", usage.prompt_tokens, "at", PRICE_IN, "USD per million")
print("output tokens:", usage.completion_tokens, "at", PRICE_OUT, "USD per million")
print(f"cost by hand:  ${by_hand:.8f}")

input tokens:  24 at 0.1 USD per million
output tokens: 7 at 0.5 USD per million
cost by hand:  $0.00000590


With our numbers:

$$\frac{24 \times 0.10 + 7 \times 0.50}{1{,}000{,}000} = 0.00000590 \text{ USD}$$

A few millionths of a dollar for one kitten name.

### The same cost from `completion_cost`

`completion_cost` takes a reply and returns its cost in US dollars. We wrote no prices for it.

In [6]:
from litellm import completion_cost

estimate = completion_cost(completion_response=litellm_response)

print(f"completion_cost: ${estimate:.8f}")

completion_cost: $0.00000590


The same number, \$0.00000590. So where did LiteLLM get its prices? It ships with a price table, `litellm.model_cost`: a
dictionary with one entry per model it knows. We print two fields of the entry for our model. The table stores
prices per token, so we multiply by a million to compare them with `models.env`.

In [7]:
entry = litellm.model_cost[MODEL]

print("input, USD per million tokens: ", round(entry["input_cost_per_token"] * 1_000_000, 4))
print("output, USD per million tokens:", round(entry["output_cost_per_token"] * 1_000_000, 4))

input, USD per million tokens:  0.1
output, USD per million tokens: 0.5


The table's prices for this model are 0.1 and 0.5 US dollars per million tokens, the same as
the two in `models.env`. That is why the two costs agree.

**Important:** `completion_cost` is an estimate, and the bill comes from the vendor. The estimate is the token
counts of the reply times a list price in a table. It is wrong when the table is out of date, and it does not include
a discount in your contract. We now hold two copies of the price list, ours in `models.env` and LiteLLM's,
and neither is the vendor's invoice. Use the estimate to compare models and to watch spending as it happens; settle
accounts from the invoice.

**Checkpoint B.** The cost by hand and `completion_cost` print the same number (ours: \$0.00000590). The table's
prices for the model match `models.env`.

---

## Part C: Ask several models the same question, side by side

So far one model has named the kitten. Which model to use is a choice we will want to try out: another model may cost less or answer better, and another vendor's model may suit the job better. With LiteLLM, trying another model means changing
one string.

`models.env` names a second OpenAI model for this comparison, in `MODEL_COMPARE`. We put the two model strings in
a list. Both start with `openai/`, so both use the one key we have.

If `.env` also holds a `GEMINI_API_KEY`, a Gemini model joins the list. Its string starts with `gemini/`, and
LiteLLM reads that vendor's key from the environment variable `GEMINI_API_KEY`. Without that key the cell says so
and carries on with the two OpenAI models.

In [8]:
models = ["openai/" + MODEL, "openai/" + settings["MODEL_COMPARE"]]

GEMINI_KEY = os.environ.get("GEMINI_API_KEY") or local.get("GEMINI_API_KEY")

if GEMINI_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_KEY
    models.append("gemini/" + settings["GEMINI_MODEL"])
else:
    print("No GEMINI_API_KEY in .env: comparing the two OpenAI models only.")

print(models)

['openai/gpt-6-luna', 'openai/gpt-4o-mini', 'gemini/gemini-3.5-flash-lite']


Our `.env` holds a Gemini key, so our list has three model strings: the main model (`MODEL_DEFAULT`), the second
OpenAI model (`MODEL_COMPARE`) and the Gemini model.

### Send the same call to every model in the list

The loop makes the call of Part A once for each model in the list. Between one pass and the next, only `model`
changes.

One argument is new: `drop_params=True`. Reasoning effort is a setting that some models do not have. Asked to send
it to such a model, LiteLLM stops with an error (`UnsupportedParamsError`) and sends nothing. With
`drop_params=True` it leaves the argument out for a model that does not take it, and sends it to the others.

For each reply we print the name, the reply's type, its token counts and its cost from `completion_cost`.

In [9]:
for model in models:
    response = litellm.completion(
        model=model,
        messages=kitten,
        reasoning_effort=EFFORT,
        drop_params=True,
    )

    print(model)
    print("  name:         ", response.choices[0].message.content.strip())
    print("  type:         ", type(response).__name__)
    print("  input tokens: ", response.usage.prompt_tokens)
    print("  output tokens:", response.usage.completion_tokens)
    print(f"  cost:          ${completion_cost(completion_response=response):.8f}")

openai/gpt-6-luna
  name:          Smokey McClawface
  type:          ModelResponse
  input tokens:  24
  output tokens: 10
  cost:          $0.00000740


openai/gpt-4o-mini
  name:          Whiskerina
  type:          ModelResponse
  input tokens:  25
  output tokens: 4
  cost:          $0.00000615


gemini/gemini-3.5-flash-lite
  name:          Ascender
  type:          ModelResponse
  input tokens:  19
  output tokens: 2
  cost:          $0.00001070


One name for the kitten from each model: *Smokey McClawface*, *Whiskerina* and *Ascender*. Read the blocks
line against line:

- **`type`** is `ModelResponse` every time, also for the Gemini model. Every reply has the same shape and was
  read with the same lines.
- **The token counts differ** (24, 25 and 19 input tokens for the same question), because each model splits the
  same text into tokens in its own way.
- **The costs differ:** \$0.00000740, \$0.00000615 and \$0.00001070. Each model has its own price per token, and
  the replies have different lengths.

**Key lesson:** to ask another model, we changed one string. We imported no other vendor's library, we built no
other kind of request, and we read every reply with the same lines.

**Notice** where the costs come from. The prices in `models.env` are those of `MODEL_DEFAULT` only, so the
arithmetic of Part B would be wrong for the other models. `completion_cost` looks each model up in LiteLLM's
table.

One name from each model says nothing about which model is better. To choose between models, run them on a set of
real questions and check the answers.

**Checkpoint C.** One block per model in the list: a name, the type `ModelResponse`, token counts and a cost. The
costs differ from model to model.

---

## Part D: Receive a reply piece by piece with `stream=True`

One name arrives in a second. A longer answer takes several, and a person in front of a chat page should not look
at an empty screen all that time. With `stream=True` the reply arrives in small pieces as the model writes it, and
we can show each piece at once.

We ask for five names this time, each with a reason, so there is something to watch.

What changes in the code:

- `stream=True` makes `litellm.completion` return at once, with a **stream** in place of the finished reply: an
  object we loop over, which hands us one **chunk** at a time.
- A chunk has `choices[0].delta` where a full reply has `choices[0].message`. The delta holds only the new piece
  of text, in `delta.content`.
- `or ""` is there because a chunk can carry no text (`None`); the first and the last one do.
- `print(piece, end="", flush=True)` prints the piece without starting a new line, and shows it immediately.

We also keep every piece in a list, to look at them afterwards.

In [10]:
IDEAS = "Suggest five names for a grey kitten who climbs the curtains. One per line, each with a few words on why."

stream = litellm.completion(
    model="openai/" + MODEL,
    messages=[{"role": "user", "content": IDEAS}],
    reasoning_effort=EFFORT,
    stream=True,
)

pieces = []
for chunk in stream:
    piece = chunk.choices[0].delta.content or ""
    pieces.append(piece)
    print(piece, end="", flush=True)

M

isty

 —

 a

 soft

 grey

... (52 outputs left out)

 loves

 to

 hang

.

Five names with their reasons. If you ran the cell yourself, you saw the lines grow word by word. In a saved
notebook only the finished text is left, so we look at the pieces themselves.

In [11]:
print("pieces:", len(pieces))
print(pieces[:10])

pieces: 69
['', 'M', 'isty', ' —', ' a', ' soft', ' grey', ' name', ' for', ' a']


The reply came in 69 pieces. The first ten show what a piece is: an empty string first, then one word, dash
or part of a word at a time (`'M'`, `'isty'`, `' —'`, `' a'`). A piece that follows a space carries that space at its
front. Joined together in
order, the pieces are the reply.

This loop is the same for every vendor LiteLLM supports. Each vendor streams in its own format, and LiteLLM turns
each one into these chunks.

**Checkpoint D.** Five names print, and the list of pieces holds a few dozen short strings (ours: 69).
The number of pieces changes from run to run.

---

## Part E: Let the model ask for a tool that our own code runs

Say we have settled on six names we like, and we cannot choose. We could let chance decide. First we ask the model
to do it.

In [12]:
NAMES = ["Biscuit", "Pixel", "Mango", "Captain Whiskers", "Noodle", "Zorro"]

# "1. Biscuit, 2. Pixel, ...": the names with their numbers, as one line of text
numbered = ", ".join(f"{number}. {name}" for number, name in enumerate(NAMES, start=1))
print(numbered)

response = litellm.completion(
    model="openai/" + MODEL,
    messages=[{"role": "user", "content": f"My six name ideas: {numbered}. Pick one at random. Reply with the name only."}],
    reasoning_effort=EFFORT,
)

print("the model picked:", response.choices[0].message.content)

1. Biscuit, 2. Pixel, 3. Mango, 4. Captain Whiskers, 5. Noodle, 6. Zorro


the model picked: Pixel


The model picked Pixel. From one pick we cannot tell whether it was
random, so run the cell a few more times. We asked nine times while writing this lab, and the
model picked Noodle eight times. The ninth answer was Pixel, in this saved run.

**Notice:** a model has no dice. It writes the text that is most likely to follow the request, and for this list
that is the same name again and again. A fair pick is a job for code.

### A dice function, and its description for the model

`roll_dice` is plain Python: `random.randint(1, sides)` returns a whole number from 1 to `sides`, each equally
likely. Its first line prints a message, so that we can see when it really runs.

In [13]:
import random


def roll_dice(sides: int) -> int:
    """Roll one die and return the number it shows."""
    print(f"🎲 TOOL CALLED: roll_dice({sides})")
    return random.randint(1, sides)


print(roll_dice(6))

🎲 TOOL CALLED: roll_dice(6)
6


The line `TOOL CALLED`, then a number from 1 to 6 (6 when we ran it).

The model never sees this Python. What we send is a description of the function, a **tool schema**, in the form
Chat Completions expects:

- `name`: what the model writes to ask for this tool.
- `description`: what the tool is for. The model decides from this text whether to ask for it.
- `parameters`: the arguments, as a JSON Schema. Here one whole number called `sides`.
- `strict: True`: the model's arguments must match the schema exactly.

In [14]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "roll_dice",
            "description": "Roll one die and return the number it shows.",
            "parameters": {
                "type": "object",
                "properties": {"sides": {"type": "integer", "description": "How many sides the die has"}},
                "required": ["sides"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

### The model asks, through LiteLLM

We ask again, this time for a roll of the die, and pass `tools=tools`. We print the reply's text and its
`tool_calls`, because this time the reply may not be text.

In [15]:
dice_messages = [{"role": "user", "content": f"My six name ideas: {numbered}. Roll a die to choose one, then tell me my kitten's name."}]

first = litellm.completion(
    model="openai/" + MODEL,
    messages=dice_messages,
    tools=tools,
    reasoning_effort=EFFORT,
)

message = first.choices[0].message
print("text:      ", message.content)
print("tool calls:", [(call.function.name, call.function.arguments) for call in message.tool_calls or []])

text:       None
tool calls: [('roll_dice', '{"sides":6}')]


The text is `None`, and there is one tool call: the name `roll_dice` and the arguments `{"sides":6}`, a string of
JSON that the model wrote.

**Notice** what did not print: no `TOOL CALLED` line. The model asked for the tool, and LiteLLM delivered the
request to us. Nothing has run.

### Our code runs the function and sends the result back

Three steps, all in our own code:

1. `dice_messages.append(...)` adds the model's request to the conversation. `model_dump(exclude_none=True)` turns
   the message object into a plain dictionary and leaves out its empty fields.
2. For each tool call, `json.loads` turns the arguments string into a dictionary, and `roll_dice(**arguments)` runs
   the function with it. `**arguments` passes `{"sides": 6}` as `sides=6`.
3. The result goes back as a message with the role `tool`, carrying the id of the call it answers. Its content is
   text, so we wrap the number in `str(...)`.

Then we call the model again with the longer conversation.

In [16]:
import json

dice_messages.append(message.model_dump(exclude_none=True))

for call in message.tool_calls:
    arguments = json.loads(call.function.arguments)
    result = roll_dice(**arguments)
    print("the die shows", result, "which is", NAMES[result - 1])
    dice_messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

final = litellm.completion(
    model="openai/" + MODEL,
    messages=dice_messages,
    tools=tools,
    reasoning_effort=EFFORT,
)

print(final.choices[0].message.content)

🎲 TOOL CALLED: roll_dice(6)
the die shows 1 which is Biscuit


Your kitten’s name is **Biscuit**!


Now `TOOL CALLED` prints, from our own line `roll_dice(**arguments)`. The die showed 1, which is
Biscuit in our list, and the model's reply names Biscuit. Run the cell that asks and this one again, and the die will most likely show another number: the choice
now comes from `random.randint`.

**Key lesson:** LiteLLM carried two requests to the model and two replies back. It did not run `roll_dice`, and it
never will. Which tools exist, whether a call is allowed, and running it all stay in our code, whichever vendor's
model wrote the request.

We ran the tool round once, by hand. An agent repeats it in a loop until a reply comes back with no tool call;
notebook 2 writes that loop.

**Checkpoint E.** The first reply has no text and one tool call for `roll_dice` with `{"sides":6}`, and no
`TOOL CALLED` line. After our code runs the function, the reply names the kitten with the number the die showed.

---

## Part F: Call a model by a name of our own with a `Router`

Every call so far named a vendor and a model: `"openai/" + MODEL`. A program with fifty such calls has to change
fifty lines when the model changes, and each line names the vendor we buy from. A `Router` keeps the vendor and the model
in one place. Our code asks for a name we made up, and the Router looks up the deployment behind that name.

We choose two names for two kinds of work: `quick` for small jobs and `careful` for jobs that need the stronger
model. `quick` will stand for the second OpenAI model of Part C (`MODEL_COMPARE`), and `careful` for the main model
(`MODEL_DEFAULT`). The Router gets a `model_list` with one entry per name. LiteLLM calls an entry a **deployment**. Each has
three parts:

| Field | What it holds |
|---|---|
| `model_name` | the name our code asks for |
| `litellm_params` | what LiteLLM really calls for that name: the arguments of `litellm.completion` (the vendor and model, and for `careful` the reasoning effort) |
| `model_info` | facts about the deployment. We set `id`, our own label for it, which comes back with every reply |

In [17]:
from litellm import Router

model_list = [
    {
        "model_name": "quick",
        "litellm_params": {"model": "openai/" + settings["MODEL_COMPARE"]},
        "model_info": {"id": "quick-on-openai"},
    },
    {
        "model_name": "careful",
        "litellm_params": {"model": "openai/" + MODEL, "reasoning_effort": EFFORT},
        "model_info": {"id": "careful-on-openai"},
    },
]

router = Router(model_list=model_list)

# What the router holds: one line per deployment
for deployment in router.registry().all():
    print(deployment["model_name"], "->", deployment["litellm_params"]["model"], "| id:", deployment["model_info"]["id"])

quick -> openai/gpt-4o-mini | id: quick-on-openai
careful -> openai/gpt-6-luna | id: careful-on-openai


Creating the Router sent nothing to a model. `router.registry().all()` returns the deployments the Router holds,
as dictionaries, and the loop printed one line for each: the name, the model behind it and its id.

**Notice:** the two names stand for two different models. Change `MODEL_COMPARE` in `models.env` and `quick`
follows; the code that asks for `quick` does not change. Only `careful`'s deployment carries the reasoning effort,
because the model behind `quick` does not take that setting.

![Our code asks the Router for the name quick. The Router holds two deployments, quick and careful, each pointing at a model. A dotted arrow from quick to careful is the fallback of Part G.](resources/lab21_router.png)

Light green is our own code, the dark boxes are models, white is LiteLLM. The dotted arrow belongs to Part G.

### Ask the Router for `quick`

`router.completion` takes the same arguments as `litellm.completion`. The `model` is now our own name, and
the reasoning effort is no longer in the call: a deployment carries the settings its model needs.

A reply from a Router has the usual fields. It also tells us which deployment wrote it, in
`response._hidden_params["model_id"]`. `_hidden_params` is a dictionary that LiteLLM attaches to every reply for
facts that have no place in OpenAI's reply shape.

In [18]:
response = router.completion(model="quick", messages=kitten)

print(response.choices[0].message.content)
print("model:     ", response.model)
print("deployment:", response._hidden_params["model_id"])

Whiskerby
model:      gpt-4o-mini-2024-07-18
deployment: quick-on-openai


A name for the kitten (*Whiskerby* this time). `model` is the vendor's model that answered, written with its
full version name, and `deployment` is the
id we gave: `quick-on-openai`. Our call said `quick` and nothing else.

**Checkpoint F.** The Router lists `quick` and `careful` with the model behind each. The call to `quick` prints a
name, the model that answered and `deployment: quick-on-openai`.

---

## Part G: Keep the answer coming when a deployment cannot be reached

A vendor's service is sometimes down. A model can be overloaded, or a rate limit is reached. What does our call get back then?

### A call to an address where nothing answers

We cannot switch OpenAI off, so we imitate the failure. `api_base` is the address LiteLLM sends a call to. We set
it to `http://127.0.0.1:9/v1`, a port on this machine where no program listens, so every request is refused
at once. We call it the dead address. The call is wrapped in `try` and `except`, so that the error prints as one line.

In [19]:
NOWHERE = "http://127.0.0.1:9/v1"

try:
    litellm.completion(model="openai/" + MODEL, messages=kitten, api_base=NOWHERE)
except Exception as error:
    print(type(error).__name__, "|", error)

InternalServerError | litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.


`InternalServerError | litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.`

LiteLLM files the failure under its own error class, `InternalServerError`; the words at the end, *Connection
error*, are what happened. OpenAI's own client calls the same failure `APIConnectionError`.

A connection error, and no name. A program that calls one vendor at one address stops here.

### A second deployment that answers when the first fails

A Router can hold a **fallback**: a second name to try when a call to the first one fails.
`fallbacks=[{"quick": ["careful"]}]` reads: when `quick` fails, try `careful`.

⚠️ We break `quick` on purpose. Its deployment gets `api_base: NOWHERE`, the address from the cell above.
`careful` stays as it was in Part F. This is a test of the fallback, and the broken address is its only purpose.

In [20]:
broken_list = [
    {
        "model_name": "quick",
        "litellm_params": {"model": "openai/" + settings["MODEL_COMPARE"], "api_base": NOWHERE},
        "model_info": {"id": "quick-on-openai"},
    },
    model_list[1],
]

fallback_router = Router(model_list=broken_list, fallbacks=[{"quick": ["careful"]}])

We ask for `quick`, with the same line as in Part F. This time we also print the model, how many fallbacks the Router
tried
(LiteLLM keeps that count in `_hidden_params["additional_headers"]`) and how long the call took.

Before you run it: which deployment id will print?

In [21]:
import time

start = time.time()
response = fallback_router.completion(model="quick", messages=kitten)

print(response.choices[0].message.content)
print("model:              ", response.model)
print("deployment:         ", response._hidden_params["model_id"])
print("fallbacks attempted:", response._hidden_params["additional_headers"]["x-litellm-attempted-fallbacks"])
print("seconds:            ", round(time.time() - start, 1))

Smokey Climber
model:               gpt-6-luna
deployment:          careful-on-openai
fallbacks attempted: 1
seconds:             6.9


The kitten has a name: *Smokey Climber*. Read the four lines under it:

- **`model`** is the main model. The answer came from the model behind `careful`, not from the one behind `quick`.
- **`deployment: careful-on-openai`.** We asked for `quick`, and `careful` wrote the answer.
- **`fallbacks attempted: 1`.** The Router moved to another name once.
- **`seconds: 6.9`.** Most of that time went on the Router trying `quick` again before it gave up on it: by
  default it retries a failed call twice, with a short wait in between.

**Notice** what the calling code saw: a normal reply. There was no error to catch. That is what a fallback is for,
and it is also a risk. If we did not print the deployment id, we would not know that `quick` is down, or that our
quick jobs now run on the model we set aside for careful ones. When you use fallbacks, record which deployment
answered.

**Important:** in Parts F to H both deployments are OpenAI models, reached with one key. That
protects us when one deployment fails (a wrong address, a limit on one model). It does not help when the vendor
itself is down. For that the fallback has to be at another vendor, and Part I does it for readers who have a
second key.

**Checkpoint G.** The direct call to the dead address prints a connection error. Through the Router with a
fallback, the same question gets a name, `deployment: careful-on-openai` and `fallbacks attempted: 1`, after some seconds of retries (ours: 6.9).

---

## Part H: Put the same names behind one address with the LiteLLM gateway

A `Router` lives inside one Python program. A company has many programs, some of them not written in Python, and
it does not want the vendor's key copied into each of them. The **LiteLLM gateway** (its documentation also calls
it the proxy) is the Router as a server:

- it runs as a program of its own and listens at an address;
- it holds the vendor's key, and keeps the names and the fallbacks in one configuration file;
- every program calls it the way it would call OpenAI, with a key that the gateway issues.

### The configuration file

The gateway reads a YAML file. We write it from this notebook, so that the model names still come from
`models.env`, and print it. The file goes in `outputs/`, a folder beside this notebook that git ignores.

In [22]:
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
CONFIG_FILE = OUTPUTS / "gateway_config.yaml"

config = f"""model_list:
  - model_name: quick
    litellm_params:
      model: openai/{settings["MODEL_COMPARE"]}
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: quick-on-openai
  - model_name: careful
    litellm_params:
      model: openai/{MODEL}
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: careful-on-openai

router_settings:
  fallbacks:
    - quick: [careful]

general_settings:
  master_key: lab-21-local-key
"""

CONFIG_FILE.write_text(config)
print(config)

model_list:
  - model_name: quick
    litellm_params:
      model: openai/gpt-4o-mini
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: quick-on-openai
  - model_name: careful
    litellm_params:
      model: openai/gpt-6-luna
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: careful-on-openai

router_settings:
  fallbacks:
    - quick: [careful]

general_settings:
  master_key: lab-21-local-key



Read the file block by block:

- **`model_list`** is Part F's list written as YAML, with two differences. Each deployment names where its key
  comes from (`api_key`), and `careful` has no reasoning effort line: the calls in this part go to `quick`. Both
  deployments work this time; nothing is broken.
- **`api_key: os.environ/OPENAI_API_KEY`** tells the gateway to read the vendor's key from its own environment
  variable of that name. The key itself is not in the file.
- **`router_settings`** holds the fallback of Part G, in YAML: when `quick` fails, try `careful`.
- **`master_key`** is the key of the gateway itself. A program that calls the gateway sends this key, and never
  the vendor's. We made this one up for a gateway that runs only on this machine. A real gateway gets a long
  random key from an environment variable.

### Start the gateway

Installing LiteLLM also installed a command, `litellm`. In a terminal, in this lab's folder, we would type:

```bash
uv run litellm --config outputs/gateway_config.yaml --host 127.0.0.1 --port 4000
```

(`uv run` starts a command in the repository's environment. With the environment activated, type `litellm ...`
alone.) The command keeps running until we stop it with `Ctrl+C`.

From a notebook we start the same program with Python's `subprocess.Popen`, which starts a command and lets the
notebook carry on while it runs:

- The loop at the top finds a free port. It starts at 4000, LiteLLM's usual port. `connect_ex` returns 0 when
  some program already answers at a port; the loop then says so and tries the next number. `PORT` and `GATEWAY`
  hold the result, and every later cell uses them.
- The command is a list: this kernel's Python (`sys.executable`), then `-m litellm.proxy.proxy_cli`, which is what
  the `litellm` command runs, then the config file, the host and the port.
- `--host 127.0.0.1` makes the gateway answer only programs on this machine. Without it the gateway listens on
  every network the machine is connected to.
- `stdout=log, stderr=subprocess.STDOUT` sends everything the gateway prints to `outputs/gateway.log`. Open that
  file when the gateway does not start.
- The gateway gets the vendor's key from this notebook's environment, where the setup cell put it.

If the cell prints a port other than 4000, use that number in the terminal command above and in the page
addresses further down.

In [23]:
import socket
import subprocess
import sys

# The first free port, starting at LiteLLM's usual 4000
PORT = 4000
while True:
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", PORT)) != 0:
            break
    print("port", PORT, "is in use, trying the next one")
    PORT += 1

GATEWAY = f"http://127.0.0.1:{PORT}"
GATEWAY_KEY = "lab-21-local-key"

with open(OUTPUTS / "gateway.log", "w") as log:
    gateway = subprocess.Popen(
        [sys.executable, "-m", "litellm.proxy.proxy_cli", "--config", str(CONFIG_FILE), "--host", "127.0.0.1", "--port", str(PORT)],
        stdout=log,
        stderr=subprocess.STDOUT,
    )

print("gateway starting on port", PORT)

gateway starting on port 4000


Ours took port 4000, which was free. When 4000 is busy (a LiteLLM gateway running in Docker usually listens there), the cell prints
`port 4000 is in use, trying the next one` and tries 4001.

The gateway needs about ten seconds to start. We wait for it with the tool we will use to call it: the plain
OpenAI client.

`gateway_client` is an ordinary `OpenAI` client with two things changed: `base_url` is the gateway's address, and
`api_key` is the gateway's key. The loop asks the gateway for its list of models once a second. `max_retries=0`
stops the OpenAI client from trying again by itself (by default it retries a failed request twice), so each pass
of the loop is one attempt. While the gateway
is still starting, the client raises `APIConnectionError` and we wait; when it answers, we have the list.
(`gateway.poll()` is `None` as long as the gateway's process is alive.)

In [24]:
from openai import APIConnectionError

gateway_client = OpenAI(base_url=GATEWAY + "/v1", api_key=GATEWAY_KEY, timeout=60, max_retries=0)

while True:
    assert gateway.poll() is None, "The gateway stopped while starting. Read outputs/gateway.log."
    try:
        names = [model.id for model in gateway_client.models.list()]
        break
    except APIConnectionError:
        time.sleep(1)

print("the gateway is up and offers:", names)

the gateway is up and offers: ['quick', 'careful']


The gateway is up, and what it offers are our two names, `quick` and `careful`. A program that asks this gateway for its models sees only these two names, not the vendor behind them.

### Call the gateway with the plain OpenAI client

This is the call of Part A again, with the client pointed at the gateway and the model set to one of our names.
We pass no reasoning effort, because the model behind `quick` does not take one.

In [25]:
answer = gateway_client.chat.completions.create(
    model="quick",
    messages=kitten,
)

print(answer.choices[0].message.content)

Sable


A name for the kitten, *Sable*, and this cell did not use the `litellm` library at all. The program that asks
needs only something that can make an OpenAI-style call: the OpenAI client in any language, or a framework that
takes a `base_url`. The vendor's key stayed with the gateway.

### What the gateway adds to a reply

Through the Router we read the deployment id from `_hidden_params`. A reply from the gateway is plain HTTP, so the
gateway puts the same facts in the reply's **headers**: lines of `name: value` that travel with every HTTP reply,
beside its body. The OpenAI client shows them when we call through `with_raw_response`.
`raw.parse()` turns the raw reply into the usual object, so that we can print its text too. The cost header is a
number in scientific notation, so the last line also prints it as dollars with eight decimals.

In [26]:
raw = gateway_client.chat.completions.with_raw_response.create(
    model="quick",
    messages=kitten,
)

print(raw.parse().choices[0].message.content)
for header in ["x-litellm-model-group", "x-litellm-model-id", "x-litellm-attempted-fallbacks", "x-litellm-response-cost"]:
    print(header + ":", raw.headers.get(header))
print("the cost in dollars:", f"${float(raw.headers['x-litellm-response-cost']):.8f}")

Nimbus
x-litellm-model-group: quick
x-litellm-model-id: quick-on-openai
x-litellm-attempted-fallbacks: 0
x-litellm-response-cost: 4.35e-06
the cost in dollars: $0.00000435


Four of the headers the gateway adds:

| Header | What it says | In our run |
|---|---|---|
| `x-litellm-model-group` | the name that answered | `quick` |
| `x-litellm-model-id` | the deployment that answered | `quick-on-openai` |
| `x-litellm-attempted-fallbacks` | how many fallbacks were tried | `0` |
| `x-litellm-response-cost` | the gateway's estimate of the cost, in US dollars | `4.35e-06`, which is \$0.00000435 |

The cost is the same kind of estimate as `completion_cost` in Part B, worked out by the gateway for every request
that passes through it. This reply (*Nimbus*) came from the model behind `quick`, with its own token
counts and its own price per token, so its cost is not the one of Part B.
Notebook 2 reads these headers for every request an agent makes.

### The gateway's pages

The gateway is running now, so this is the moment to open its pages in a browser, before the next code cell stops
it.

- **`http://127.0.0.1:4000/`** is the gateway's API page. If the start cell printed another port, use that number here
and in the next address. It lists every address the gateway answers at, from
  `/v1/chat/completions` to the ones that manage keys and budgets.
- **`http://127.0.0.1:4000/ui`** is the admin page, where a team adds models, issues keys, sets budgets and reads
  what was spent. Its sign-in needs a database behind the gateway. Ours has none, so the sign-in answers
  *Not connected to DB!* The picture shows the admin page of a gateway that has one.

![The first screen of the LiteLLM admin page. A menu on the left lists Virtual Keys, Playground, Models and Endpoints, Usage, Logs, Teams and more. The middle shows a button Create New Virtual Key and four counters: Spend, Total Requests, Total Tokens and Average cost per request, all at zero.](resources/litellm_admin_page.png)

*Screenshot to add: the gateway's API page at `http://127.0.0.1:4000/` from this notebook's own run.*

The **Going further** section of notebook 2 links to the steps for starting a gateway with a database, for
readers who have Docker.

### Stop the gateway

`terminate()` asks the gateway's process to stop, and `wait()` waits until it has. Its port is free again afterwards.

In [27]:
gateway.terminate()
gateway.wait()

print("gateway stopped")

gateway stopped


⚠️ Always run this cell before you close the notebook or restart its kernel. A gateway that is left running keeps
its port, and the next start has to take the next one (see Troubleshooting).

**Checkpoint H.** The config file prints with `quick`, `careful`, the fallback and the gateway's key. The gateway
offers `['quick', 'careful']`. The plain OpenAI client gets a kitten name from it, and the reply's text prints with headers that
name the deployment `quick-on-openai`, 0 fallbacks and a cost. The last cell prints `gateway stopped`.

---

## Part I (optional): Fall back to another vendor's model

In Part G the backup was a second OpenAI model, so it fails when OpenAI itself is down. A backup that survives a
vendor's outage is a model at another vendor. Part C already sent the kitten question to a Gemini model; here a Gemini
model becomes the fallback. With LiteLLM that is a change to one deployment: another prefix in the model string, and that vendor's key.

⚠️ This part needs a second key, `GEMINI_API_KEY`, in `.env`. It is optional: without the key (Part C looked for it) the cell
prints a message and skips, and nothing in notebook 2 depends on it. The Gemini model's name is `GEMINI_MODEL` in
`models.env`.

The Router below keeps the broken `quick` of Part G. Its `careful` now points at a Gemini model: the model string
starts with `gemini/`, and `api_key` passes the Gemini key.

In [28]:
if not GEMINI_KEY:
    print("No GEMINI_API_KEY in .env: skipping this optional part.")
else:
    two_vendor_list = [
        broken_list[0],
        {
            "model_name": "careful",
            "litellm_params": {"model": "gemini/" + settings["GEMINI_MODEL"], "api_key": GEMINI_KEY},
            "model_info": {"id": "careful-on-gemini"},
        },
    ]
    two_vendor_router = Router(model_list=two_vendor_list, fallbacks=[{"quick": ["careful"]}])

    response = two_vendor_router.completion(model="quick", messages=kitten)

    print(response.choices[0].message.content)
    print("model:     ", response.model)
    print("deployment:", response._hidden_params["model_id"])
    print(f"cost:       ${completion_cost(completion_response=response):.8f}")

Velcro
model:      gemini-3.5-flash-lite
deployment: careful-on-gemini
cost:       $0.00001070


With the key in place, the saved run shows a name (*Velcro*), then a Gemini model on the `model:` line and
`deployment: careful-on-gemini`. The calling line still asked for `quick`. The cost line (\$0.00001070) comes
from the same price table as in Part C.

**Checkpoint I.** With a Gemini key: a name, a Gemini model and `deployment: careful-on-gemini`. Without one: the message that the part is skipped.

---

## What you can do now

- Send a call with `litellm.completion`, say what the prefix of the model name does, and read the reply with the
  code that reads an OpenAI reply (Part A).
- Price a reply by hand and with `completion_cost`, and say why both are estimates (Part B).
- Ask several models the same question in one loop, and price each reply (Part C).
- Receive a reply as a stream of pieces (Part D).
- Let the model ask for a tool through LiteLLM, and run the tool in your own code (Part E).
- Give models names of your own with a `Router`, and read which deployment answered (Part F).
- Add a fallback, test it by breaking a deployment on purpose, and say what a fallback hides (Part G).
- Write a gateway's config file, start and stop the gateway, call it with the plain OpenAI client and read its
  headers (Part H).
- Make the fallback a model at another vendor (Part I, optional).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.` outside Part G | LiteLLM could not reach the address it was given | check the network, and that no `api_base` from Part G is left in the call or the deployment |
| `BadRequestError: litellm.BadRequestError: You passed in model=fast. There are no healthy deployments for this model` | the Router was asked for a name that is not in its `model_list` | ask for one of the names the Router printed (`quick`, `careful`) |
| `port 4000 is in use, trying the next one` from the start cell | another program is using that port: a LiteLLM gateway in Docker, or a gateway of this lab that was never stopped (the kernel was restarted before the stop cell ran) | Nothing to fix: the cell takes the next free port. To stop a leftover gateway of this lab, in a terminal: `pkill -f litellm.proxy.proxy_cli` on Linux or a Mac; on Windows end the `python` process that runs it in the Task Manager |
| `UnsupportedParamsError: litellm.UnsupportedParamsError: openai does not support parameters: ['reasoning_effort'], for model=...` | the call passed `reasoning_effort` to a model that has no such setting | add `drop_params=True` to the call (Part C), or leave the argument out |
| `AssertionError: The gateway stopped while starting. Read outputs/gateway.log.` | the gateway could not start; the last lines of the log say why | open `outputs/gateway.log`; a missing package means the environment needs `uv sync` |
| `Error code: 400 - {'error': {'message': "/chat/completions: Invalid model name passed in model=...` from the gateway | the call named a model the gateway does not offer, for example the vendor's own model name | use one of the gateway's names; `gateway_client.models.list()` shows them |
| *Not connected to DB!* on the admin page's sign-in | the admin page needs a database behind the gateway | expected here; notebook 2's Going further links to the steps for a gateway with a database |

## Challenge

1. **Roll the die while `quick` is down.** Repeat Part E's tool round through `fallback_router`: ask for `quick`
   with `tools=tools`, run `roll_dice` yourself, send the result back, again to `quick`. Print the deployment id of
   both replies. Expected: one `TOOL CALLED` line, a reply that names the kitten with the number the die showed,
   and `careful-on-openai` twice. The tool round works on whichever deployment answers, because the tool runs in
   our code.
2. **Price the two replies the fallback wrote.** Add up `completion_cost` for the two replies of task 1, and work out the same sum by hand
   from the token counts and the prices in `models.env` (they apply here, because `careful` is `MODEL_DEFAULT`). Expected: the two sums agree, and the total is a small
   fraction of a cent. Then say what this estimate cannot contain: the attempts on `quick` that failed. Here they
   cost nothing, because nothing answered at that address.

The worked solution, run with its outputs, is in `solved/21a_meet_litellm_solution.ipynb`.

## Resources

- [LiteLLM: getting started](https://docs.litellm.ai/docs/): `completion`, the providers and the model name
  prefixes
- [Output](https://docs.litellm.ai/docs/completion/output): the fields of a `ModelResponse`
- [Completion token usage and cost](https://docs.litellm.ai/docs/completion/token_usage): `completion_cost` and the
  price table
- [Streaming](https://docs.litellm.ai/docs/completion/stream): `stream=True` and the chunks
- [Function calling](https://docs.litellm.ai/docs/completion/function_call): tools through LiteLLM
- [Router](https://docs.litellm.ai/docs/routing): `model_list`, deployments and how a Router chooses
- [Fallbacks](https://docs.litellm.ai/docs/proxy/reliability): fallbacks, retries and cooldowns
- [Gateway quick start](https://docs.litellm.ai/docs/proxy/quick_start) and
  [the config file](https://docs.litellm.ai/docs/proxy/configs): starting the gateway and what `config.yaml` holds
- [Response headers](https://docs.litellm.ai/docs/proxy/response_headers): every `x-litellm-` header
- [Gemini through LiteLLM](https://docs.litellm.ai/docs/providers/gemini): the `gemini/` prefix and its key

Next: [notebook 2, LiteLLM at the Chinook desk](21b_litellm_at_the_desk.ipynb).